# 39. HS4 Fixed-Label Macro-F1 Validation

38번의 Inner 검증에서는 각 seed의 검증 데이터에 실제 등장한 HS4 클래스만으로 Macro-F1을 계산했습니다. 이번 실험은 검증 표본에서 일부 클래스가 빠져도 항상 전체 5개 HS4 클래스를 기준으로 Macro-F1을 계산해 튜닝 기준을 통일합니다.

- 동일 평가 모집단: 195건, HS4 5개, 제품군 29개
- Outer 평가: Leave-One-Product-Family-Out (LOPO)
- Inner 튜닝: 반복 Stratified Group K-Fold (5 folds × 5 seeds)
- 비교: Baseline, Direct only, Exclusion only, Direct + Exclusion, Fixed36
- Inner Macro-F1: 항상 전체 HS4 5개 클래스로 계산
- 1차 지표: HS4 Macro-F1 / 2차 지표: Accuracy
- Google Drive 미사용

## 실험 설계 및 38번과의 차이

38번의 Inner 튜닝은 seed별 검증 데이터에 등장한 클래스만 Macro-F1 계산에 포함했습니다. 따라서 일부 검증 반복에서 희소 HS4 클래스가 빠지면, 해당 클래스의 성능이 점수에 반영되지 않을 수 있었습니다.

39번은 모델, 데이터, Outer LOPO 분할, Inner StratifiedGroupKFold 분할 및 후보 설정을 38번과 동일하게 유지합니다. 유일한 핵심 변경은 Inner Macro-F1 계산 시 `labels=hs4_labels`를 고정하여 항상 5개 HS4 클래스를 모두 평가하는 것입니다. 검증에 나타나지 않은 클래스는 해당 반복에서 F1=0으로 처리됩니다.

이 변경은 클래스 누락이 있는 검증 반복의 점수를 보수적으로 평가합니다. 다만 Inner 검증 커버리지 자체를 완전히 해결하는 것은 아니므로, 커버리지 감사 결과와 함께 해석합니다. 38번과 39번의 비교를 통해 기존 튜닝 결과가 클래스 구성에 얼마나 민감했는지 확인합니다.

38번 백업의 임베딩 캐시가 있다면 업로드 후 복구할 수 있습니다. 캐시가 없으면 임베딩을 다시 생성합니다.

## 0. 환경 설정

In [ ]:
!pip -q install -U sentence-transformers scipy kiwipiepy scikit-learn
import re, zipfile, requests
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
WORK=Path('/content/39_hs4_fixed_macro_validation')
WORK.mkdir(parents=True,exist_ok=True)
BASE='https://raw.githubusercontent.com/lin1118-code/HS-complete-product/feature/explanatory-note-integration/'
print(WORK)

## 1. 실험 29와 동일한 데이터 파이프라인

In [ ]:
src=requests.get(BASE+'29_discriminative_features_experiment.ipynb').json()
for i in [3,5,7]: exec(''.join(src['cells'][i]['source']),globals())
print('평가 표본:',len(eval_df),'| HS6:',len(labels),'| 제품군:',eval_df.product_family.nunique(),'| HS4:',eval_df.hs4.nunique())
assert len(eval_df)==195 and len(labels)==11 and eval_df.product_family.nunique()==29 and eval_df.hs4.nunique()==5
hs4_labels=sorted(eval_df.hs4.astype(str).unique())
print('HS4 labels:',hs4_labels)
display(eval_df.groupby('hs4').agg(n=('hs4','size'),families=('product_family','nunique')))
display(pd.crosstab(eval_df.product_family,eval_df.hs4))

## 2. 해설서 근거 채널 생성

In [ ]:
kb=pd.read_json(BASE+'hs_knowledge_base.jsonl',lines=True).fillna('')
for c in ['knowledge_id','hs_code','rule_type','document_text','source_sheet','source_page_or_line']:
    if c not in kb: kb[c]=''
    kb[c]=kb[c].astype(str).str.strip()
kb['hs_digits']=kb.hs_code.map(lambda x:re.sub(r'\D','',x))
def clean_kb_text(x):
    lines=[]
    for line in str(x).splitlines():
        if re.match(r'^(규칙_ID|HS코드|자료출처|류|적용범위)\s*:',line.strip()): continue
        lines.append(line.strip())
    return ' '.join(x for x in lines if x)
kb['embedding_text']=kb.document_text.map(clean_kb_text)
direct_rows=[]
exclusion_rows=[]
for _,r in kb.iterrows():
    typ=r.rule_type
    own=r.hs_digits[:4]
    if own in hs4_labels and r.embedding_text and '제외' not in typ:
        direct_rows.append({**r.to_dict(),'candidate_hs4':own,'channel':'direct'})
    if '제외' in typ and r.embedding_text:
        refs=sorted(set(re.findall(r'(?:제\s*)?(\d{4})\s*호',r.document_text)))
        for h in refs:
            if h[:4] in hs4_labels: exclusion_rows.append({**r.to_dict(),'candidate_hs4':h[:4],'channel':'exclusion'})
direct_docs=pd.DataFrame(direct_rows).drop_duplicates(['knowledge_id','candidate_hs4']).reset_index(drop=True)
exclusion_docs=pd.DataFrame(exclusion_rows).drop_duplicates(['knowledge_id','candidate_hs4']).reset_index(drop=True)
print('KB:',len(kb),'| Direct:',len(direct_docs),'| Exclusion links:',len(exclusion_docs))
display(pd.concat([direct_docs.groupby('candidate_hs4').size().rename('direct_docs'),exclusion_docs.groupby('candidate_hs4').size().rename('exclusion_links')],axis=1).reindex(hs4_labels).fillna(0).astype(int))
display(exclusion_docs[['knowledge_id','candidate_hs4','rule_type','source_page_or_line','document_text']])

## 3. 임베딩 생성 및 캐시

In [ ]:
CACHE=WORK/'embeddings_39.npz'
if CACHE.exists():
    z=np.load(CACHE,allow_pickle=True)
    query_emb,product_emb,ruling_emb,kb_direct_emb,kb_exclusion_emb=z['query_emb'],z['product_emb'],z['ruling_emb'],z['kb_direct_emb'],z['kb_exclusion_emb']
else:
    sbert=SentenceTransformer('intfloat/multilingual-e5-base')
    def encode_query(x): return sbert.encode(['query: '+str(v) for v in x],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
    def encode_passage(x): return sbert.encode(['passage: '+str(v) for v in x],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
    query_emb=encode_query(eval_df.query_text.tolist())
    product_emb=encode_passage(eval_df.query_text.tolist())
    ruling_emb=encode_passage(eval_df.ruling_text.tolist())
    kb_direct_emb=encode_passage(direct_docs.embedding_text.tolist()) if len(direct_docs) else np.empty((0,768),dtype=float)
    kb_exclusion_emb=encode_passage(exclusion_docs.embedding_text.tolist()) if len(exclusion_docs) else np.empty((0,768),dtype=float)
    np.savez_compressed(CACHE,query_emb=query_emb,product_emb=product_emb,ruling_emb=ruling_emb,kb_direct_emb=kb_direct_emb,kb_exclusion_emb=kb_exclusion_emb)
print(query_emb.shape,product_emb.shape,ruling_emb.shape,kb_direct_emb.shape,kb_exclusion_emb.shape)

## 4. 순위 함수와 고정 검색 캐시

In [ ]:
def aggregate_hs4(scores,codes,method='top3_mean'):
    d=pd.DataFrame({'hs4':np.asarray(codes).astype(str),'score':np.asarray(scores,dtype=float)})
    if method=='max': out=d.groupby('hs4').score.max()
    elif method=='mean': out=d.groupby('hs4').score.mean()
    else: out=d.groupby('hs4').score.apply(lambda x:x.nlargest(min(3,len(x))).mean())
    return out.sort_values(ascending=False)
def rrf_two(a,b,k=60,w=1.0):
    pa={str(c):i+1 for i,c in enumerate(a.index)}
    pb={str(c):i+1 for i,c in enumerate(b.index)}
    return pd.Series({c:1/(k+pa.get(c,99999))+w/(k+pb.get(c,99999)) for c in hs4_labels}).sort_values(ascending=False)
def rrf_ablation(base,direct=None,exclusion=None,wd=0.0,we=0.0,k=60):
    ranks=[base]
    weights=[1.0]
    if direct is not None: ranks.append(direct); weights.append(wd)
    if exclusion is not None: ranks.append(exclusion); weights.append(we)
    pos=[{str(c):i+1 for i,c in enumerate(r.index)} for r in ranks]
    return pd.Series({c:sum(w/(k+p.get(c,99999)) for w,p in zip(weights,pos)) for c in hs4_labels}).sort_values(ascending=False)
def rank_kb(scores,docs,method):
    vals={}
    for c in hs4_labels:
        x=np.asarray(scores)[docs.candidate_hs4.astype(str).to_numpy()==c]
        if len(x)==0: vals[c]=-1.0
        elif method=='max': vals[c]=float(np.max(x))
        else: vals[c]=float(np.mean(np.sort(x)[-min(3,len(x)):]))
    return pd.Series(vals).sort_values(ascending=False)
def baseline_rank(train_idx,q):
    codes=eval_df.iloc[train_idx].hs4.astype(str).to_numpy()
    rp=aggregate_hs4(product_emb[train_idx]@query_emb[q],codes,'top3_mean')
    rr=aggregate_hs4(ruling_emb[train_idx]@query_emb[q],codes,'top3_mean')
    return rrf_two(rp,rr,w=1.0)
METHODS=['max','top3_mean']
WEIGHTS=[0.0,0.5,1.0,2.0]
direct_rank_cache={}
exclusion_rank_cache={}
for q in range(len(eval_df)):
    ds=kb_direct_emb@query_emb[q] if len(direct_docs) else np.array([])
    es=kb_exclusion_emb@query_emb[q] if len(exclusion_docs) else np.array([])
    direct_rank_cache[q]={m:rank_kb(ds,direct_docs,m) for m in METHODS}
    exclusion_rank_cache[q]={m:rank_kb(es,exclusion_docs,m) for m in METHODS}
splits=list(LeaveOneGroupOut().split(eval_df,eval_df.hs4,groups=eval_df.product_family))
print('Outer LOPO:',len(splits))
print('Direct docs:',len(direct_docs),'| Exclusion links:',len(exclusion_docs))

## 5. 반복 Stratified Group K-Fold Inner 튜닝

In [ ]:
INNER_SEEDS=[11,23,37,53,71]
INNER_SPLITS=5
CONFIGS_B=[(m,w) for m in METHODS for w in WEIGHTS]
CONFIGS_C=[(m,w) for m in METHODS for w in WEIGHTS]
CONFIGS_D=[(md,me,wd,we) for md in METHODS for me in METHODS for wd in WEIGHTS for we in WEIGHTS]
FIXED36=('top3_mean','max',0.5,0.5)
def predict_variant(base,q,variant,cfg):
    if variant=='B':
        m,w=cfg
        return str(rrf_ablation(base,direct_rank_cache[q][m],None,w,0).index[0])
    if variant=='C':
        m,w=cfg
        return str(rrf_ablation(base,None,exclusion_rank_cache[q][m],0,w).index[0])
    md,me,wd,we=cfg
    return str(rrf_ablation(base,direct_rank_cache[q][md],exclusion_rank_cache[q][me],wd,we).index[0])
def build_inner_repeats(outer_train):
    inner_df=eval_df.iloc[outer_train].reset_index(drop=True)
    y=inner_df.hs4.astype(str).to_numpy()
    groups=inner_df.product_family.to_numpy()
    all_labels=set(hs4_labels)
    repeats=[]
    audit=[]
    for seed in INNER_SEEDS:
        splitter=StratifiedGroupKFold(n_splits=INNER_SPLITS,shuffle=True,random_state=seed)
        split_rows=[]
        validated=set()
        skipped=[]
        for split_no,(itr,iva) in enumerate(splitter.split(inner_df,y,groups),1):
            global_tr=outer_train[itr]
            global_va=outer_train[iva]
            train_labels=set(eval_df.iloc[global_tr].hs4.astype(str))
            valid_labels=set(eval_df.iloc[global_va].hs4.astype(str))
            if train_labels!=all_labels:
                skipped.append('|'.join(sorted(set(inner_df.iloc[iva].product_family.astype(str)))))
                continue
            rows=[]
            for q in global_va:
                rows.append({'q':int(q),'true':str(eval_df.iloc[q].hs4),'base':baseline_rank(global_tr,q)})
                validated.add(int(q))
            split_rows.append(rows)
        if not split_rows:
            raise ValueError(f'Inner seed {seed}: 유효 split이 없습니다.')
        repeats.append(split_rows)
        excluded=sorted(set(outer_train.tolist())-validated)
        excluded_families=sorted(set(eval_df.iloc[excluded].product_family.astype(str))) if excluded else []
        audit.append({'seed':seed,'valid_splits':len(split_rows),'validation_rows':len(validated),'coverage_rate':len(validated)/len(outer_train),'validated_families':len(set(eval_df.iloc[list(validated)].product_family.astype(str))),'excluded_families':'|'.join(excluded_families),'skipped_split_groups':' ; '.join(skipped)})
    return repeats,audit
def score_config(repeats,variant,cfg):
    f1s=[]; accs=[]; rows=[]; classes=[]
    for split_rows in repeats:
        yt=[]; yp=[]
        for one_split in split_rows:
            for item in one_split:
                yt.append(item['true'])
                yp.append(predict_variant(item['base'],item['q'],variant,cfg))
        present=sorted(set(yt))
        f1s.append(f1_score(yt,yp,labels=hs4_labels,average='macro',zero_division=0))
        accs.append(accuracy_score(yt,yp))
        rows.append(len(yt))
        classes.append(len(present))
    return float(np.mean(f1s)),float(np.mean(accs)),float(np.std(f1s,ddof=1)) if len(f1s)>1 else 0.0,float(np.mean(rows)),float(np.mean(classes))
def tune_variant(repeats,variant):
    configs={'B':CONFIGS_B,'C':CONFIGS_C,'D':CONFIGS_D}[variant]
    scored=[]
    for cfg in configs:
        mf,acc,sd,nrows,nclasses=score_config(repeats,variant,cfg)
        scored.append((mf,acc,cfg,sd,nrows,nclasses))
    scored.sort(key=lambda x:(x[0],x[1]),reverse=True)
    return scored[0],pd.DataFrame([{'variant':variant,'config':str(x[2]),'mean_inner_macro_f1':x[0],'mean_inner_accuracy':x[1],'std_inner_macro_f1':x[3],'mean_validation_rows_per_repeat':x[4],'mean_validated_classes':x[5]} for x in scored])
print('Inner 설정 수 | B:',len(CONFIGS_B),'C:',len(CONFIGS_C),'D:',len(CONFIGS_D),'| repeats:',len(INNER_SEEDS),'| folds:',INNER_SPLITS)

## 6. Outer LOPO 실행 (5개 Fold씩)

In [ ]:
FOLD_START=1
FOLD_END=5
def fold_path(f): return WORK/f'fold_{f:02d}.csv'
def tuning_path(f): return WORK/f'fold_{f:02d}_tuning.csv'
def coverage_path(f): return WORK/f'fold_{f:02d}_coverage.csv'
def grid_path(f): return WORK/f'fold_{f:02d}_gridsearch.csv'
for fold,(tr,te) in enumerate(splits,1):
    if not FOLD_START<=fold<=FOLD_END: continue
    if fold_path(fold).exists() and tuning_path(fold).exists() and coverage_path(fold).exists() and grid_path(fold).exists():
        print('Fold',fold,'완료 - skip')
        continue
    train_labels=set(eval_df.iloc[tr].hs4.astype(str))
    test_labels=set(eval_df.iloc[te].hs4.astype(str))
    if train_labels!=set(hs4_labels) or not test_labels.issubset(train_labels):
        raise ValueError(f'Fold {fold}: Outer 학습 라벨 커버리지 오류')
    repeats,coverage=build_inner_repeats(tr)
    selected={}
    rankings={}
    tuning_tables=[]
    for v in ['B','C','D']:
        best,table=tune_variant(repeats,v)
        selected[v]=best
        table['outer_fold']=fold
        tuning_tables.append(table)
    rows=[]
    for q in te:
        base=baseline_rank(tr,q)
        md,me,wd,we=FIXED36
        ranks={
            'Baseline_34_RRF':base,
            'Direct_Only_Tuned':rrf_ablation(base,direct_rank_cache[q][selected['B'][2][0]],None,selected['B'][2][1],0),
            'Exclusion_Only_Tuned':rrf_ablation(base,None,exclusion_rank_cache[q][selected['C'][2][0]],0,selected['C'][2][1]),
            'Direct_Plus_Exclusion_Tuned':rrf_ablation(base,direct_rank_cache[q][selected['D'][2][0]],exclusion_rank_cache[q][selected['D'][2][1]],selected['D'][2][2],selected['D'][2][3]),
            'Direct_Plus_Exclusion_Fixed36':rrf_ablation(base,direct_rank_cache[q][md],exclusion_rank_cache[q][me],wd,we)
        }
        for model,rank in ranks.items():
            rows.append({'row_index':int(q),'fold':fold,'heldout_family':eval_df.iloc[q].product_family,'true_hs4':str(eval_df.iloc[q].hs4),'model':model,'predicted_hs4':str(rank.index[0]),'top3_hs4':'|'.join(rank.index[:3].astype(str))})
    pd.DataFrame(rows).to_csv(fold_path(fold),index=False,encoding='utf-8-sig')
    tun=[]
    for v,best in selected.items():
        tun.append({'fold':fold,'variant':v,'selected_config':str(best[2]),'mean_inner_macro_f1':best[0],'mean_inner_accuracy':best[1],'std_inner_macro_f1':best[3],'mean_validation_rows_per_repeat':best[4],'mean_validated_classes':best[5]})
    pd.DataFrame(tun).to_csv(tuning_path(fold),index=False,encoding='utf-8-sig')
    pd.concat(tuning_tables,ignore_index=True).to_csv(grid_path(fold),index=False,encoding='utf-8-sig')
    pd.DataFrame(coverage).assign(outer_fold=fold).to_csv(coverage_path(fold),index=False,encoding='utf-8-sig')
    print(f'Fold {fold}/{len(splits)} | B={selected["B"][2]} | C={selected["C"][2]} | D={selected["D"][2]}')
    print('평균 Inner 커버리지:',round(np.mean([x['coverage_rate'] for x in coverage]),3))
print('현재 실행 구간 완료')

## 7. 진행률 및 중간 백업

In [ ]:
done=sorted(int(p.stem.split('_')[1]) for p in WORK.glob('fold_[0-9][0-9].csv'))
missing=sorted(set(range(1,len(splits)+1))-set(done))
print(f'완료 Fold: {len(done)}/{len(splits)} | 누락: {missing}')
BACKUP='/content/39_fold_backup.zip'
with zipfile.ZipFile(BACKUP,'w',zipfile.ZIP_DEFLATED) as z:
    for pattern in ['fold_[0-9][0-9].csv','fold_[0-9][0-9]_tuning.csv','fold_[0-9][0-9]_coverage.csv','fold_[0-9][0-9]_gridsearch.csv']:
        for p in WORK.glob(pattern): z.write(p,p.name)
    if CACHE.exists(): z.write(CACHE,CACHE.name)
from google.colab import files
files.download(BACKUP)

## 8. 중간 백업 복구 (새 런타임에서만 실행)

In [ ]:
from google.colab import files
uploaded=files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z: z.extractall(WORK)
        print('복구 완료:',name)

## 9. 통합 평가 및 최종 결과 ZIP

In [ ]:
files_list=sorted([p for p in WORK.glob('fold_[0-9][0-9].csv')])
done=sorted(int(p.stem.split('_')[1]) for p in files_list)
missing=sorted(set(range(1,len(splits)+1))-set(done))
print(f'완료 Fold: {len(done)}/{len(splits)} | 누락: {missing}')
if missing: raise ValueError('미완료 Outer Fold가 있습니다.')
pred=pd.concat([pd.read_csv(p) for p in files_list],ignore_index=True)
if pred.duplicated(['row_index','model']).any(): raise ValueError('중복 예측 존재')
def metric(g):
    yt=g.true_hs4.astype(str); yp=g.predicted_hs4.astype(str)
    return pd.Series({'n':len(g),'HS4_Macro_F1':f1_score(yt,yp,labels=hs4_labels,average='macro',zero_division=0),'HS4_Accuracy':accuracy_score(yt,yp)})
summary=pred.groupby('model').apply(metric,include_groups=False).reset_index().sort_values(['HS4_Macro_F1','HS4_Accuracy'],ascending=False)
class_rows=[]
for model,g in pred.groupby('model'):
    p,r,f,s=precision_recall_fscore_support(g.true_hs4.astype(str),g.predicted_hs4.astype(str),labels=hs4_labels,zero_division=0)
    for c,pv,rv,fv,sv in zip(hs4_labels,p,r,f,s): class_rows.append({'model':model,'HS4':c,'support':int(sv),'precision':pv,'recall':rv,'F1':fv})
class_metrics=pd.DataFrame(class_rows)
tuning=pd.concat([pd.read_csv(p) for p in sorted(WORK.glob('fold_[0-9][0-9]_tuning.csv'))],ignore_index=True)
coverage=pd.concat([pd.read_csv(p) for p in sorted(WORK.glob('fold_[0-9][0-9]_coverage.csv'))],ignore_index=True)
gridsearch=pd.concat([pd.read_csv(p) for p in sorted(WORK.glob('fold_[0-9][0-9]_gridsearch.csv'))],ignore_index=True)
paired=[]
base=pred[pred.model=='Baseline_34_RRF'][['row_index','true_hs4','predicted_hs4']].rename(columns={'predicted_hs4':'baseline_pred'})
for model,g in pred[pred.model!='Baseline_34_RRF'].groupby('model'):
    x=base.merge(g[['row_index','predicted_hs4']],on='row_index')
    paired.append({'model':model,'changed_predictions':int((x.baseline_pred!=x.predicted_hs4).sum()),'baseline_wrong_to_correct':int(((x.baseline_pred!=x.true_hs4)&(x.predicted_hs4==x.true_hs4)).sum()),'baseline_correct_to_wrong':int(((x.baseline_pred==x.true_hs4)&(x.predicted_hs4!=x.true_hs4)).sum()),'baseline_and_model_wrong':int(((x.baseline_pred!=x.true_hs4)&(x.predicted_hs4!=x.true_hs4)&(x.baseline_pred!=x.predicted_hs4)).sum())})
paired=pd.DataFrame(paired)
display(summary.round(4))
display(class_metrics.sort_values(['model','HS4']).round(4))
display(tuning.sort_values(['variant','fold']))
display(coverage.sort_values(['outer_fold','seed']))
display(paired)
OUT=WORK/'final'; OUT.mkdir(exist_ok=True)
pred.to_csv(OUT/'39_oof_predictions.csv',index=False,encoding='utf-8-sig')
summary.to_csv(OUT/'39_model_comparison.csv',index=False,encoding='utf-8-sig')
class_metrics.to_csv(OUT/'39_hs4_class_metrics.csv',index=False,encoding='utf-8-sig')
tuning.to_csv(OUT/'39_selected_configs_by_fold.csv',index=False,encoding='utf-8-sig')
coverage.to_csv(OUT/'39_inner_validation_coverage.csv',index=False,encoding='utf-8-sig')
gridsearch.to_csv(OUT/'39_inner_gridsearch_scores.csv',index=False,encoding='utf-8-sig')
paired.to_csv(OUT/'39_paired_prediction_changes.csv',index=False,encoding='utf-8-sig')
for model,g in pred.groupby('model'):
    pd.DataFrame(confusion_matrix(g.true_hs4.astype(str),g.predicted_hs4.astype(str),labels=hs4_labels),index=hs4_labels,columns=hs4_labels).to_csv(OUT/f'39_confusion_{model}.csv',encoding='utf-8-sig')
with zipfile.ZipFile('/content/39_hs4_fixed_macro_validation_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob('*.csv'): z.write(p,p.name)
print('저장 완료: /content/39_hs4_fixed_macro_validation_results.zip')

## 결과 해석 기준

1. 38번의 고정36 결합 모델이 37번/36번과 비슷한 성능을 보이는지 확인합니다.
2. 반복 Stratified Group K-Fold 튜닝 모델이 고정36 설정보다 안정적으로 높은지 비교합니다.
3. Direct / Exclusion 단독 모델은 가중치가 0으로 선택되는 빈도와 Outer 성능을 함께 확인합니다.
4. 각 Outer Fold의 Inner 검증 커버리지, seed별 검증 행 수, 제외 제품군을 확인합니다.
5. 모델 선택은 Macro-F1 우선으로 하되 Accuracy와 클래스별 Recall도 함께 고려합니다.

Inner 검증 커버리지가 여전히 낮거나 seed별 편차가 크면, 해당 결과를 최종 모델 선택의 근거로 단독 사용하지 않습니다.